# Análise de redes de depedências de pacotes Python

In [ ]:
import pandas as pd
import requests
from json import load
from operator import itemgetter
import networkx as nx

from pyvis.network import Network
from matplotlib import cm
from matplotlib.colors import rgb2hex

## Download requirements

In [ ]:
with open("C:\\Users\\angel\\OneDrive\\Documentos\\MBA\\TCC\\Dataset\\Downloads\\full_scam.json", "r") as f:
	download_ranking = pd.DataFrame(load(f))

In [ ]:
deps = []
for index, row in download_ranking.head(100).iterrows():
	current_deps = {}
	response = requests.get(f"https://api.deps.dev/v3/systems/pypi/packages/{row['package_name']}/versions/{row['version']}")
	if response.status_code != 200:
		continue
	is_default, is_deprecated, attestations = itemgetter("isDefault", "isDeprecated", "attestations")(response.json())
	is_verified = False
	if attestations:
		attestation, = attestations
		is_verified = attestation.get("verified", False)
	current_deps |= {
		"package_name": row["package_name"],
		"version": row["version"],
		"is_default": is_default,
		"is_deprecated": is_deprecated,
		"is_verified": is_verified
	}
	response = requests.get(f"https://api.deps.dev/v3/systems/pypi/packages/{row['package_name']}/versions/{row['version']}:requirements")
	if response.status_code != 200:
		continue
	requirements = response.json()["pypi"]["dependencies"]
	current_deps |= {
		"requirements": requirements
	}
	deps.append(current_deps)

In [ ]:
# from json import dump
# with open("C:\\Users\\angel\\OneDrive\\Documentos\\MBA\\TCC\\Dataset\\Downloads\\full_deps.json", "w") as f:
# 	dump(deps, f)

In [ ]:
with open("C:\\Users\\angel\\OneDrive\\Documentos\\MBA\\TCC\\Dataset\\Downloads\\full_deps.json", "r") as f:
	deps = load(f)
deps_df = pd.DataFrame(deps)

In [ ]:
df_deps = pd.DataFrame(deps)

In [ ]:
ranking_graph = df_deps.merge(download_ranking, on=["package_name", "version"], how="left")

In [ ]:
reqs = pd.json_normalize(df_deps.explode("requirements")["requirements"])

In [ ]:
edges = pd.concat(
    [
        df_deps[["package_name"]].reset_index(drop=True),
        reqs.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
df_deps = df_deps.explode("requirements")

In [ ]:
df_deps =  pd.concat([df_deps.drop('requirements', axis=1), df_deps['requirements'].apply(pd.Series)], axis=1)

In [ ]:
df_deps = df_deps.rename(columns={'package_name': 'source'})

In [ ]:
df_deps = df_deps.rename(columns={'projectName': 'target'})

In [ ]:
df_deps_ = df_deps.dropna(subset=['target', 'source'])

In [ ]:
G = nx.DiGraph()

In [ ]:
df_deps_[["source", "target"]].values.tolist()

In [ ]:
G.add_edges_from(zip(df_deps_["source"].to_list(), df_deps_["target"].to_list()))

In [ ]:
# nx.pagerank(G)

In [ ]:
nx.betweenness_centrality(G)

In [ ]:
nx.degree_centrality(G)

In [ ]:
G.in_degree('numpy')

In [ ]:
G.out_degree('numpy')

In [ ]:
nx.weakly_connected_components(G)

In [ ]:

net = Network(directed=True, notebook=True)
net.from_nx(G)

net.show_buttons()
net.write_html("graph__.html")

In [ ]:
ego = nx.ego_graph(
    G,
    "numpy",
    radius=1,
    undirected=True
)

In [ ]:
ego_net = Network(
    height="900px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    directed=True
)

for node in ego.nodes():

    if node == "numpy":

        ego_net.add_node(
            node,
            color="#FF4444",
            size=60,
            title="Biblioteca analisada"
        )

    else:

        ego_net.add_node(
            node,
            color="#4FC3F7",
            size=20
        )

for source, target in ego.edges():
    ego_net.add_edge(source, target)

ego_net.barnes_hut(
    gravity=-8000,
    spring_length=250
)

ego_net.write_html("ego_numpy.html")

In [ ]:

# =========================
# MÉTRICAS
# =========================

# Betweenness → tamanho do nó
betweenness = nx.betweenness_centrality(G)

# Closeness → cor do nó
closeness = nx.closeness_centrality(G)

# =========================
# NORMALIZAÇÃO
# =========================

max_bet = max(betweenness.values())
max_close = max(closeness.values())

# mapa de cores
cmap = cm.get_cmap("Reds")

# =========================
# PYVIS
# =========================

net = Network(
    height="750px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    directed=True
)

# física do grafo
net.barnes_hut()

# =========================
# ADICIONAR NÓS
# =========================

for node in G.nodes():

    # tamanho baseado em betweenness
    size = 15 + (betweenness[node] / max_bet) * 50 if max_bet > 0 else 15

    # cor baseada em closeness
    normalized_close = closeness[node] / max_close if max_close > 0 else 0
    color = rgb2hex(cmap(normalized_close))

    # tooltip
    title = f"""
    <b>{node}</b><br>
    Betweenness: {betweenness[node]:.4f}<br>
    Closeness: {closeness[node]:.4f}
    """

    net.add_node(
        node,
        label=node,
        size=size,
        color=color,
        title=title
    )

# =========================
# ADICIONAR ARESTAS
# =========================

for source, target in G.edges():
    net.add_edge(
        source,
        target,
        arrows="to"
    )

# =========================
# EXPORTAR
# =========================

net.write_html("grafo_vulnerabilidade.html")

In [ ]:
bet = nx.betweenness_centrality(G)

top_nodes = sorted(
    bet,
    key=bet.get,
    reverse=True
)[:100]

G_filtered = G.subgraph(top_nodes)

In [ ]:
threshold = 0.01

top_nodes = [
    n for n, c in bet.items()
    if c > threshold
]

In [ ]:
import math

In [ ]:

# =========================
# MÉTRICAS
# =========================

# Betweenness → tamanho do nó
betweenness = nx.betweenness_centrality(G_filtered)

# Closeness → cor do nó
closeness = nx.closeness_centrality(G_filtered)

# =========================
# NORMALIZAÇÃO
# =========================

max_bet = max(betweenness.values())
max_close = max(closeness.values())

# mapa de cores
cmap = cm.get_cmap("Reds")

# =========================
# PYVIS
# =========================

net = Network(
    height="750px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    directed=True
)

# física do grafo
net.barnes_hut()

# =========================
# ADICIONAR NÓS
# =========================

for node in G_filtered.nodes():

    # tamanho baseado em betweenness
    size = 20 + math.log1p(closeness[node] * 10000) * 15
    # size = 15 + (betweenness[node] / max_bet) * 50 if max_bet > 0 else 15

    # cor baseada em closeness
    normalized_close = betweenness[node] / max_close if max_close > 0 else 0
    color = rgb2hex(cmap(normalized_close))

    # tooltip
    title = f"""
    <b>{node}</b><br>
    Betweenness: {betweenness[node]:.4f}<br>
    Closeness: {closeness[node]:.4f}
    """

    net.add_node(
        node,
        label=node,
        size=size,
        color=color,
        title=title
    )

# =========================
# ADICIONAR ARESTAS
# =========================

for source, target in G_filtered.edges():
    net.add_edge(
        source,
        target,
        arrows="to"
    )

# =========================
# EXPORTAR
# =========================

net.write_html("grafo_filtrado_vulnerabilidade.html")

In [ ]:
# =====================================
# EXEMPLO DE GRAFO
# A -> B significa:
# A depende de B
# =====================================

nodes_to_keep = [
    node for node in G.nodes()
    if G.degree(node) > 1
]

G = G.subgraph(nodes_to_keep).copy()

# =====================================
# MÉTRICAS
# =====================================

# Tamanho dos nós
betweenness = nx.betweenness_centrality(G)

# Cor dos nós
closeness = nx.closeness_centrality(G)

max_close = max(closeness.values())

# =====================================
# PYVIS
# =====================================

net = Network(
    height="900px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    directed=True
)

# Layout melhor para grafos complexos
net.force_atlas_2based()

# mapa de cores
cmap = cm.get_cmap("Reds")

# =====================================
# PACOTES VULNERÁVEIS (EXEMPLO)
# =====================================

vulnerable_packages = [
    "urllib3",
    "jinja2"
]

# =====================================
# ADICIONAR NÓS
# =====================================

for node in G.nodes():

    # =================================
    # COR = closeness
    # =================================

    normalized_close = closeness[node] / max_close
    color = rgb2hex(cmap(normalized_close))

    # =================================
    # TAMANHO = betweenness (log scale)
    # =================================

    size = 20 + math.log1p(
        betweenness[node] * 10000
    ) * 20

    # =================================
    # LABELS APENAS NOS IMPORTANTES
    # =================================

    label = node if size > 45 else ""

    # =================================
    # BORDA PARA VULNERÁVEIS
    # =================================

    if node in vulnerable_packages:
        border_color = "yellow"
        border_width = 5
    else:
        border_color = color
        border_width = 1

    # =================================
    # TOOLTIP
    # =================================

    title = f"""
    <b>{node}</b><br>
    Betweenness: {betweenness[node]:.4f}<br>
    Closeness: {closeness[node]:.4f}
    """

    # =================================
    # ADD NODE
    # =================================

    net.add_node(
        node,
        label=label,
        size=size,
        color={
            "background": color,
            "border": border_color
        },
        borderWidth=border_width,
        title=title
    )

# =====================================
# ADICIONAR ARESTAS
# =====================================

for source, target in G.edges():

    net.add_edge(
        source,
        target,
        arrows="to",
        color="#888888"
    )

# =====================================
# CONFIGURAÇÕES EXTRAS
# =====================================

net.set_options("""
var options = {
  "nodes": {
    "font": {
      "size": 20
    }
  },
  "edges": {
    "smooth": {
      "type": "dynamic"
    }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.01,
      "springLength": 180,
      "springConstant": 0.08
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  }
}
""")

# =====================================
# EXPORTAR HTML
# =====================================

net.write_html("grafo_filtrado_reformulado_vulnerabilidades.html")

In [ ]:

# =========================
# MÉTRICAS
# =========================

# Betweenness → tamanho do nó
betweenness = nx.betweenness_centrality(G_filtered)

# Closeness → cor do nó
closeness = nx.closeness_centrality(G_filtered)


# =====================================
# NORMALIZAÇÕES
# =====================================

max_close = max(closeness.values())
max_bet = max(betweenness.values())

# =====================================
# ADICIONAR NÓS
# =====================================

for node in G_filtered.nodes():

    # =================================
    # TAMANHO = closeness
    # =================================

    close_norm = closeness[node] / max_close

    # aumenta contraste visual
    size = 15 + (close_norm ** 2) * 80

    # =================================
    # COR = betweenness
    # =================================

    bet_norm = betweenness[node] / max_bet if max_bet > 0 else 0

    # potência aumenta contraste
    bet_norm = bet_norm ** 0.4

    color = rgb2hex(cmap(bet_norm))

    # =================================
    # LABELS
    # =================================

    label = node if size > 40 else ""


    # =================================
    # TOOLTIP
    # =================================

    title = f"""
    <b>{node}</b><br>
    Closeness: {closeness[node]:.4f}<br>
    Betweenness: {betweenness[node]:.4f}
    """

    # =================================
    # ADD NODE
    # =================================

    net.add_node(
        node,
        label=label,
        size=size,
        color={
            "background": color
        },
        title=title
    )

# =========================
# ADICIONAR ARESTAS
# =========================

for source, target in G_filtered.edges():
    net.add_edge(
        source,
        target,
        arrows="to"
    )

# =========================
# EXPORTAR
# =========================

net.write_html("grafo_filtrado_vulnerabilidade_.html")

In [ ]:

cmap = cm.get_cmap("inferno_r")
# =========================
# MÉTRICAS
# =========================

# Betweenness → tamanho do nó
betweenness = nx.betweenness_centrality(G)

# Closeness → cor do nó
closeness = nx.closeness_centrality(G)


# =====================================
# NORMALIZAÇÕES
# =====================================

max_close = max(closeness.values())
max_bet = max(betweenness.values())

# =====================================
# ADICIONAR NÓS
# =====================================

for node in G.nodes():

    # =================================
    # TAMANHO = closeness
    # =================================

    close_norm = closeness[node] / max_close

    # aumenta contraste visual
    size = 15 + (close_norm ** 2) * 80

    # =================================
    # COR = betweenness
    # =================================

    bet_norm = betweenness[node] / max_bet if max_bet > 0 else 0

    # potência aumenta contraste
    bet_norm = bet_norm ** 0.4

    color = rgb2hex(cmap(bet_norm))

    # =================================
    # LABELS
    # =================================

    label = node if size > 40 else ""


    # =================================
    # TOOLTIP
    # =================================

    title = f"""
    <b>{node}</b><br>
    Closeness: {closeness[node]:.4f}<br>
    Betweenness: {betweenness[node]:.4f}
    """

    # =================================
    # ADD NODE
    # =================================

    net.add_node(
        node,
        label=label,
        size=size,
        color={
            "background": color
        },
        title=title
    )

# =========================
# ADICIONAR ARESTAS
# =========================

for source, target in G_filtered.edges():
    net.add_edge(
        source,
        target,
        arrows="to"
    )

# =========================
# EXPORTAR
# =========================

net.write_html("grafo_vulnerabilidade__inr.html")

In [ ]:
from pyvis.network import Network
import networkx as nx
import matplotlib.cm as cm
from matplotlib.colors import rgb2hex
import numpy as np

# =====================================================
# EXEMPLO:
# A -> B significa:
# A depende de B
# =====================================================

edges = [
    ("pandas", "numpy"),
    ("scikit-learn", "numpy"),
    ("matplotlib", "numpy"),
    ("seaborn", "matplotlib"),
    ("streamlit", "pandas"),
    ("streamlit", "numpy"),
    ("tensorflow", "numpy"),
    ("keras", "tensorflow"),
    ("requests", "urllib3"),
    ("boto3", "urllib3"),
    ("botocore", "urllib3"),
    ("flask", "jinja2"),
    ("dash", "flask"),
    ("fastapi", "pydantic"),
    ("langchain", "pydantic"),
    ("langchain", "numpy"),
    ("xgboost", "numpy"),
    ("lightgbm", "numpy"),
    ("plotly", "numpy"),
]

G = nx.DiGraph()
G.add_edges_from(zip(df_deps_["source"].to_list(), df_deps_["target"].to_list()))

# =====================================================
# REDUZIR HAIRY BALL
# remove nós periféricos
# =====================================================

nodes_to_keep = [
    n for n in G.nodes()
    if G.degree(n) > 1
]

G = G.subgraph(nodes_to_keep).copy()

# =====================================================
# MÉTRICAS
# =====================================================

# superfície de ataque
closeness = nx.closeness_centrality(G)

# criticidade estrutural
betweenness = nx.betweenness_centrality(G)

# =====================================================
# NORMALIZAÇÕES
# =====================================================

close_vals = np.array(list(closeness.values()))
bet_vals = np.array(list(betweenness.values()))

close_max = close_vals.max()
bet_max = bet_vals.max()

# colormap
cmap = cm.get_cmap("viridis")

# =====================================================
# PYVIS
# =====================================================

net = Network(
    height="900px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    directed=True
)

# layout melhor para evitar hairy ball
net.force_atlas_2based(
    gravity=-50,
    central_gravity=0.005,
    spring_length=180,
    spring_strength=0.08
)

# =====================================================
# ADICIONAR NÓS
# =====================================================

for node in G.nodes():

    # =============================================
    # TAMANHO = closeness
    # =============================================

    close_norm = closeness[node] / close_max

    # amplifica diferença visual
    size = 15 + (close_norm ** 2.5) * 120

    # =============================================
    # COR = betweenness
    # =============================================

    if bet_max > 0:
        bet_norm = betweenness[node] / bet_max
    else:
        bet_norm = 0

    # aumenta contraste do inferno
    bet_norm = min(1, bet_norm * 4)
    bet_norm = bet_norm ** 0.35

    color = rgb2hex(cmap(bet_norm))

    # =============================================
    # LABELS SOMENTE IMPORTANTES
    # =============================================

    label = node if size > 45 else ""

    # =============================================
    # TOOLTIP
    # =============================================

    title = f"""
    <b>{node}</b><br>
    Degree: {G.degree(node)}<br>
    Betweenness: {betweenness[node]:.5f}<br>
    Closeness: {closeness[node]:.5f}
    """

    # =============================================
    # ADD NODE
    # =============================================

    net.add_node(
        node,
        label=label,
        size=size,
        color=color,
        title=title
    )

# =====================================================
# ADICIONAR ARESTAS
# =====================================================

for source, target in G.edges():

    net.add_edge(
        source,
        target,
        arrows="to",
        width=1.5,
        color="#666666"
    )

# =====================================================
# CONFIGURAÇÕES VISUAIS
# =====================================================

net.set_options("""
var options = {
  "nodes": {
    "font": {
      "size": 22
    }
  },
  "edges": {
    "smooth": {
      "type": "dynamic"
    }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -120,
      "centralGravity": 0.005,
      "springLength": 220,
      "springConstant": 0.04
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  }
}
""")

# =====================================================
# EXPORTAR
# =====================================================

net.write_html("grafo_vulnerabilidades.html")

In [ ]:
bet = nx.betweenness_centrality(G)

In [ ]:
top_libs = sorted(
    bet,
    key=bet.get,
    reverse=True
)[:5]

In [ ]:
for lib in top_libs:

    ego = nx.ego_graph(
        G,
        lib,
        radius=1,
        undirected=True
    )

    net = Network(
        height="900px",
        width="100%",
        bgcolor="#111111",
        font_color="white",
        directed=True
    )

    for node in ego.nodes():

        if node == lib:

            net.add_node(
                node,
                color="#FF4444",
                size=60
            )

        else:

            net.add_node(
                node,
                color="#4FC3F7",
                size=20
            )

    for edge in ego.edges():
        net.add_edge(*edge)

    net.write_html(f"ego_{lib}.html")

In [ ]:
def gerar_dependentes_principais(
    G,
    biblioteca,
    metrica=None,
    top_n=20,
    arquivo=None
):
    
    if arquivo is None:
        arquivo = f"{biblioteca}_top_dependentes.html"

    dependentes = list(G.predecessors(biblioteca))

    if metrica is None:
        metrica = dict(G.degree())

    dependentes = sorted(
        dependentes,
        key=lambda x: metrica.get(x, 0),
        reverse=True
    )[:top_n]

    net = Network(
        height="900px",
        width="100%",
        bgcolor="#111111",
        font_color="white",
        directed=True
    )

    # nó central
    net.add_node(
        biblioteca,
        color="#FF4444",
        size=60,
        title=f"Biblioteca analisada: {biblioteca}"
    )

    # dependentes
    for dep in dependentes:

        net.add_node(
            dep,
            color="#4FC3F7",
            size=20,
            title=f"{dep}<br>Degree: {G.degree(dep)}"
        )

        net.add_edge(
            dep,
            biblioteca
        )

    net.repulsion(
        node_distance=250,
        spring_length=250
    )

    net.write_html(arquivo)

    return dependentes

In [ ]:
gerar_dependentes_principais(
    G,
    "numpy",
    top_n=20
)

In [ ]:
betweenness = nx.betweenness_centrality(G)

In [ ]:
gerar_dependentes_principais(
    G,
    "numpy",
    metrica=betweenness,
    top_n=20,
    arquivo="numpy_betweenness.html"
)

In [ ]:
for lib in top_libs:

    gerar_dependentes_principais(
        G,
        lib,
        metrica=betweenness,
        top_n=20
    )